<a href="https://colab.research.google.com/github/elinimuleg00-bot/30-Days-of-Python-DevOps/blob/main/Day11/Day11_time_feature_builder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Time-Series Feature Engineering Pipeline
A machine learning preprocessing script demonstrating time-series data handling in Pandas, including string-to-datetime conversion (pd.to_datetime), temporal feature extraction using .dt accessors (hour, dayofweek), lag feature generation (.shift()), and dataset serialization for MLOps pipelines.

In [ ]:
import os
import pandas as pd
import numpy as np

CSV_FILE_PATH = "raw_time_series.csv"

raw_data = """timestamp_str,server_load_pct
2026-09-01 08:00:00,45.2
2026-09-01 09:00:00,62.8
2026-09-01 10:00:00,88.1
2026-09-01 11:00:00,91.4
2026-09-01 12:00:00,55.0
2026-09-05 14:00:00,22.3
2026-09-06 15:00:00,18.7
"""

with open(CSV_FILE_PATH, "w") as file:
  file.write(raw_data)

def build_time_features(file_path):
    """Loads a time-series CSV, converts timestamps, extracts temporal features

    (hour, day_of_week, is_weekend), and generates lag features for ML pipelines.
    """
    try:
      #load dataset
      df = pd.read_csv(file_path)
      print("--- 1. RAW DATASET ---")
      print(df)

      # Step A: Convert raw string to datetime object
      df["timestamp"] = pd.to_datetime(df["timestamp_str"])

      # Step B: Extract temporal ML features using .dt accessors
      df["hour"] = df["timestamp"].dt.hour
      df["day_of_week"] = df["timestamp"].dt.dayofweek
      df["is_weekend"] = df["day_of_week"].isin([5,6]).astype(int)

      # Step C: Create Lag Features
      # .shift(1) moves values down by 1 row so the model can learn from recent history
      df["load_lag-1"] = df["server_load_pct"].shift(1)

      # Step D: Clean up non-numeric or helper columns before export
      # Drop the raw string from df bcuz ml can't calculate string(every column sent to an ML model must be a number) earlier script we didn't drop bcuz we were just printing
      ml_ready_df = df.drop(columns=["timestamp_str"])

      print("\n--- 2. TIME-FEATURE ENRICHED DATASET ---")
      print(ml_ready_df)

      # saves ml_ready to disk as csv so that the next stage of your MLOps pipeline (like model training) can load the preprocessed features directly.
      output_path = "processed_time_features.csv"
      ml_ready_df.to_csv(output_path, index=False)
      print(f"\n Time-series features exported to '{output_path}'")

    except FileNotFoundError:
      print(f"error: The file '{file_path}' was not found.")
    except Exception as e:
      print(f"An unexpected error occured: {e}")

if __name__ == "__main__":
  build_time_features(CSV_FILE_PATH)



--- 1. RAW DATASET ---
         timestamp_str  server_load_pct
0  2026-09-01 08:00:00             45.2
1  2026-09-01 09:00:00             62.8
2  2026-09-01 10:00:00             88.1
3  2026-09-01 11:00:00             91.4
4  2026-09-01 12:00:00             55.0
5  2026-09-05 14:00:00             22.3
6  2026-09-06 15:00:00             18.7

--- 2. TIME-FEATURE ENRICHED DATASET ---
   server_load_pct           timestamp  hour  day_of_week  is_weekend  \
0             45.2 2026-09-01 08:00:00     8            1           0   
1             62.8 2026-09-01 09:00:00     9            1           0   
2             88.1 2026-09-01 10:00:00    10            1           0   
3             91.4 2026-09-01 11:00:00    11            1           0   
4             55.0 2026-09-01 12:00:00    12            1           0   
5             22.3 2026-09-05 14:00:00    14            5           1   
6             18.7 2026-09-06 15:00:00    15            6           1   

   load_lag-1  
0         NaN 